# Few-Shot, Chain-of-Thought, Tree-of-Thought Notebook

> Hands-on Build It and Exercises.

## Build It

We will build a math problem solver that combines few-shot prompting, chain-of-thought reasoning, and self-consistency voting into a single pipeline. Then we will add tree-of-thought for hard problems.

The full implementation is in `code/advanced_prompting.py`. Here are the key components.

### Step 1: Few-Shot Example Store

The first component manages few-shot examples and selects the most relevant ones for a given problem.

In [ ]:
```python

GSM8K_EXAMPLES = [

    {

        "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells every egg at the farmers' market for $2. How much does she make every day at the farmers' market?",

        "reasoning": "Janet's ducks lay 16 eggs per day. She eats 3 and bakes 4, using 3 + 4 = 7 eggs. So she has 16 - 7 = 9 eggs left. She sells each for $2, so she makes 9 * 2 = $18 per day.",

        "answer": "18"

    },

    ...

]

In [ ]:
```

Each example has three parts: the question, the reasoning chain, and the final answer. The reasoning chain is what transforms a regular few-shot example into a CoT few-shot example.

### Step 2: Chain-of-Thought Prompt Builder

The prompt builder assembles a system message, few-shot examples with reasoning chains, and the target question into a single prompt.

In [ ]:
```python

def build_cot_prompt(question, examples, num_examples=3):

    system = (

        "You are a math problem solver. "

        "For each problem, show your step-by-step reasoning, "

        "then give the final numerical answer on the last line "

        "in the format: 'The answer is [number]'."

    )

    example_text = ""

    for ex in examples[:num_examples]:

        example_text += f"Q: {ex['question']}\n"

        example_text += f"A: {ex['reasoning']} The answer is {ex['answer']}.\n\n"

    user = f"{example_text}Q: {question}\nA:"

    return system, user

In [ ]:
```

The format constraint ("The answer is [number]") is critical. Without it, self-consistency cannot extract and compare answers across samples.

### Step 3: Self-Consistency Voting

Sample N reasoning paths and take the majority answer.

In [ ]:
```python

def self_consistency_solve(question, examples, client, model, n_samples=5):

    system, user = build_cot_prompt(question, examples)

    answers = []

    reasonings = []

    for _ in range(n_samples):

        response = client.chat.completions.create(

            model=model,

            messages=[

                {"role": "system", "content": system},

                {"role": "user", "content": user}

            ],

            temperature=0.7

        )

        text = response.choices[0].message.content

        reasonings.append(text)

        answer = extract_answer(text)

        if answer is not None:

            answers.append(answer)

    vote_counts = Counter(answers)

    best_answer = vote_counts.most_common(1)[0][0] if vote_counts else None

    confidence = vote_counts[best_answer] / len(answers) if best_answer else 0

    return best_answer, confidence, reasonings, vote_counts

In [ ]:
```

Temperature 0.7 is important. At temperature 0.0, all N samples would be identical, defeating the purpose. You need enough randomness for diverse reasoning paths but not so much that the model produces gibberish.

### Step 4: Tree-of-Thought Solver

For problems where linear reasoning fails, ToT explores multiple approaches and evaluates which direction is most promising.

In [ ]:
```python

def tree_of_thought_solve(question, client, model, breadth=3, depth=3):

    thoughts = generate_initial_thoughts(question, client, model, breadth)

    scored = [(t, evaluate_thought(t, question, client, model)) for t in thoughts]

    scored.sort(key=lambda x: x[1], reverse=True)

    for current_depth in range(1, depth):

        next_thoughts = []

        for thought, score in scored[:2]:

            extensions = extend_thought(thought, question, client, model, breadth)

            for ext in extensions:

                ext_score = evaluate_thought(ext, question, client, model)

                next_thoughts.append((ext, ext_score))

        scored = sorted(next_thoughts, key=lambda x: x[1], reverse=True)

    best_thought = scored[0][0] if scored else ""

    return extract_answer(best_thought), best_thought

In [ ]:
```

The evaluator is itself an LLM call. You ask the model: "On a scale of 0.0 to 1.0, how promising is this reasoning path for solving the problem?" This is the key insight of ToT -- the model evaluates its own partial solutions.

### Step 5: Full Pipeline

The pipeline combines all techniques with an escalation strategy.

In [ ]:
```python

def solve_with_escalation(question, examples, client, model):

    system, user = build_cot_prompt(question, examples)

    single_response = call_llm(client, model, system, user, temperature=0.0)

    single_answer = extract_answer(single_response)

    sc_answer, confidence, _, _ = self_consistency_solve(

        question, examples, client, model, n_samples=5

    )

    if confidence >= 0.8:

        return sc_answer, "self_consistency", confidence

    tot_answer, _ = tree_of_thought_solve(question, client, model)

    return tot_answer, "tree_of_thought", None

In [ ]:
```

The escalation logic: try cheap (single CoT) first. If self-consistency confidence is below 0.8 (less than 4 of 5 samples agree), escalate to ToT. This balances cost and accuracy -- most problems are solved cheaply, hard problems get more compute.

## Exercises

In [ ]:
1. **Measure the gap**: Take 10 GSM8K problems. Solve each with zero-shot, few-shot, zero-shot CoT, and few-shot CoT. Record accuracy for each. Which technique gives the biggest lift on your model?

2. **Example selection experiment**: For the same 10 problems, compare random example selection vs hand-picked similar examples. Measure accuracy difference. At what point does example quality matter more than example quantity?

3. **Self-consistency cost curve**: Run self-consistency with N=1, 3, 5, 7, 10 on 20 GSM8K problems. Plot accuracy vs cost (total tokens). Where is the knee of the curve for your model?

4. **Build a ReAct loop**: Extend the pipeline with a calculator tool. When the model generates a math expression, execute it with Python's `eval()` (in a sandbox) and feed the result back. Measure if tool-grounded reasoning outperforms pure CoT.

5. **ToT for creative tasks**: Adapt the Tree-of-Thought solver for a creative writing task: "Write a 6-word story that is both funny and sad." Use the LLM as evaluator. Does branching exploration produce better creative outputs than single-shot generation?